In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Test")
    .master("local[*]")
    .getOrCreate()
)

print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [2]:
# Път към CSV файла
file_path = "customers-data.csv"

# Зареждане на CSV в DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Показване на записите
df.show()

+-----------+---------------+----+---------------+----------------+-------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|
+-----------+---------------+----+---------------+----------------+-------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|
|          4|    Anna Koleva|  -5|             50|     Electronics|   2025-01-08|
|          5|           NULL|  28|           NULL|        Clothing|   2025-01-09|
|          6| Petar Dimitrov|  35|            300|           Books|   2025-01-10|
|          7|    Ivan Petrov|NULL|            120|     Electronics|   2025-01-11|
|          8|  Maria Ivanova|  32|            250|        Clothing|         NULL|
|          9|Geo

In [3]:
# Премахване на редове с липсващи стойности в name

df_clean = df.dropna(subset=["name"])
df_clean.show()

+-----------+---------------+----+---------------+----------------+-------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|
+-----------+---------------+----+---------------+----------------+-------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|
|          4|    Anna Koleva|  -5|             50|     Electronics|   2025-01-08|
|          6| Petar Dimitrov|  35|            300|           Books|   2025-01-10|
|          7|    Ivan Petrov|NULL|            120|     Electronics|   2025-01-11|
|          8|  Maria Ivanova|  32|            250|        Clothing|         NULL|
|          9|Georgi Georgiev|  40|            180|     Electronics|   2025-01-12|
+-----------+---

In [4]:
from pyspark.sql.functions import col, avg, median

# Попълване на липсващи стойности 
# Средна стойност за purchase_amount, медиана за age, Unknown за name
avg_purchase = df.select(avg(col("purchase_amount"))).first()[0]
median_age = df.select(median(col("age"))).first()[0]

df_filled = df.fillna({
    "age": median_age, 
    "purchase_amount": avg_purchase,
    "name": "Unknown"
})
df_filled.show()

+-----------+---------------+---+---------------+----------------+-------------+
|customer_id|           name|age|purchase_amount|product_category|purchase_date|
+-----------+---------------+---+---------------+----------------+-------------+
|          1|    Ivan Petrov| 25|            100|     Electronics|   2025-01-05|
|          2|  Maria Ivanova| 30|            200|        Clothing|   2025-01-06|
|          2|  Maria Ivanova| 30|            200|        Clothing|   2025-01-06|
|          3|Georgi Georgiev| 40|            150|           Books|   2025-01-07|
|          4|    Anna Koleva| -5|             50|     Electronics|   2025-01-08|
|          5|        Unknown| 28|            172|        Clothing|   2025-01-09|
|          6| Petar Dimitrov| 35|            300|           Books|   2025-01-10|
|          7|    Ivan Petrov| 30|            120|     Electronics|   2025-01-11|
|          8|  Maria Ivanova| 32|            250|        Clothing|         NULL|
|          9|Georgi Georgiev

In [29]:
from pyspark.sql.functions import col, when

# Добавяне на индикаторна колона за липсващи в purchase_date
df_filled = df_filled.withColumn(
    "missing_date_flag",
    when(col("purchase_date").isNull(), 1).otherwise(0)
)
df_filled.show()

+-----------+---------------+---+---------------+----------------+-------------+-----------------+
|customer_id|           name|age|purchase_amount|product_category|purchase_date|missing_date_flag|
+-----------+---------------+---+---------------+----------------+-------------+-----------------+
|          1|    Ivan Petrov| 25|            100|     Electronics|   2025-01-05|                0|
|          2|  Maria Ivanova| 30|            200|        Clothing|   2025-01-06|                0|
|          2|  Maria Ivanova| 30|            200|        Clothing|   2025-01-06|                0|
|          3|Georgi Georgiev| 40|            150|           Books|   2025-01-07|                0|
|          4|    Anna Koleva| -5|             50|     Electronics|   2025-01-08|                0|
|          5|        Unknown| 28|            172|        Clothing|   2025-01-09|                0|
|          6| Petar Dimitrov| 35|            300|           Books|   2025-01-10|                0|
|         

In [30]:
# Премахване на дубликати (запазва уникалните редове)
df_no_duplicates = df_filled.dropDuplicates()
df_no_duplicates.show()

+-----------+---------------+---+---------------+----------------+-------------+-----------------+
|customer_id|           name|age|purchase_amount|product_category|purchase_date|missing_date_flag|
+-----------+---------------+---+---------------+----------------+-------------+-----------------+
|          7|    Ivan Petrov| 30|            120|     Electronics|   2025-01-11|                0|
|          4|    Anna Koleva| -5|             50|     Electronics|   2025-01-08|                0|
|          5|        Unknown| 28|            172|        Clothing|   2025-01-09|                0|
|          3|Georgi Georgiev| 40|            150|           Books|   2025-01-07|                0|
|          6| Petar Dimitrov| 35|            300|           Books|   2025-01-10|                0|
|          2|  Maria Ivanova| 30|            200|        Clothing|   2025-01-06|                0|
|          1|    Ivan Petrov| 25|            100|     Electronics|   2025-01-05|                0|
|         

In [31]:
# Откриване на отрицателни възрасти
df_invalid_age = df_no_duplicates.filter((col("age") < 0) | (col("age") > 120))
df_invalid_age.show()

# Корекция на отрицателни възрасти и възрасти над 120 (с NULL)
df_corrected = df_no_duplicates.withColumn(
    "age",
    when((col("age") < 0) | (col("age") > 120), None).otherwise(col("age"))
)
df_corrected.show()

+-----------+-----------+---+---------------+----------------+-------------+-----------------+
|customer_id|       name|age|purchase_amount|product_category|purchase_date|missing_date_flag|
+-----------+-----------+---+---------------+----------------+-------------+-----------------+
|          4|Anna Koleva| -5|             50|     Electronics|   2025-01-08|                0|
+-----------+-----------+---+---------------+----------------+-------------+-----------------+

+-----------+---------------+----+---------------+----------------+-------------+-----------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|missing_date_flag|
+-----------+---------------+----+---------------+----------------+-------------+-----------------+
|          7|    Ivan Petrov|  30|            120|     Electronics|   2025-01-11|                0|
|          4|    Anna Koleva|NULL|             50|     Electronics|   2025-01-08|                0|
|          5|        Unk

In [32]:
from pyspark.sql.functions import dayofweek, month, date_format

# Създаване на нови колони: ден от седмицата и месец
df = df.withColumn("day_of_week", date_format("purchase_date", "EEEE"))
df = df.withColumn("month", date_format("purchase_date", "MMMM"))

df.show()

+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|day_of_week|  month|
+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|     Sunday|January|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|    Tuesday|January|
|          4|    Anna Koleva|  -5|             50|     Electronics|   2025-01-08|  Wednesday|January|
|          5|           NULL|  28|           NULL|        Clothing|   2025-01-09|   Thursday|January|
|          6| Petar Dimitrov|  35|            300|           Books|   2025-01-10| 

In [33]:
from pyspark.ml.feature import StringIndexer

#  Кодиране с етикети (label encoding) – всяка категория се заменя с цяло число
indexer = StringIndexer(inputCol="product_category", outputCol="category_encoded")
df_encoded = indexer.fit(df).transform(df)
df_encoded.show()

+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+----------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|day_of_week|  month|category_encoded|
+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+----------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|     Sunday|January|             1.0|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January|             0.0|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January|             0.0|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|    Tuesday|January|             2.0|
|          4|    Anna Koleva|  -5|             50|     Electronics|   2025-01-08|  Wednesday|January|             1.0|
|          5|           NULL|  28|           NUL

In [34]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# кодиране с бинарни стойности (one-hot encoding) за product_category
# 1. Индексиране, т.е. кодиране с етикети – всяка категория се заменя с цяло число
indexer = StringIndexer(inputCol="product_category", outputCol="category_encoded")
df_encoded = indexer.fit(df).transform(df)

# 2. Кодиране с бинарни стойности (one-hot encoding)
encoder = OneHotEncoder(inputCol="category_encoded", outputCol="category_ohe", dropLast=False)
df_ohe = encoder.fit(df_encoded).transform(df_encoded)

# В PySpark полученият category_ohe е вектор, не отделни колони.
df_ohe.show()

+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+----------------+-------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|day_of_week|  month|category_encoded| category_ohe|
+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+----------------+-------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|     Sunday|January|             1.0|(3,[1],[1.0])|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January|             0.0|(3,[0],[1.0])|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January|             0.0|(3,[0],[1.0])|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|    Tuesday|January|             2.0|(3,[2],[1.0])|
|          4|    Anna Koleva|  -5|             50|     Electronics|  

In [40]:
# Min–Max скалиране
from pyspark.ml.feature import MinMaxScaler, VectorAssembler

# Създаваме вектор от числовите колони
assembler = VectorAssembler(inputCols=["age", "purchase_amount"], outputCol="features", handleInvalid="skip")
df_vector = assembler.transform(df)

# Създаваме MinMaxScaler
scaler = MinMaxScaler(inputCol="features", outputCol="scaled_features")
scaler_model = scaler.fit(df_vector)
# Прилагаме скалирането 
df_scaled = scaler_model.transform(df_vector)

df_scaled.select("customer_id", "age", "purchase_amount", "scaled_features").show(truncate=False)
#df_scaled.show(truncate=False)

+-----------+---+---------------+------------------------+
|customer_id|age|purchase_amount|scaled_features         |
+-----------+---+---------------+------------------------+
|1          |25 |100            |[0.6666666666666667,0.2]|
|2          |30 |200            |[0.7777777777777778,0.6]|
|2          |30 |200            |[0.7777777777777778,0.6]|
|3          |40 |150            |[1.0,0.4]               |
|4          |-5 |50             |(2,[],[])               |
|6          |35 |300            |[0.888888888888889,1.0] |
|8          |32 |250            |[0.8222222222222223,0.8]|
|9          |40 |180            |[1.0,0.52]              |
+-----------+---+---------------+------------------------+



In [42]:
from pyspark.sql.functions import log1p
# Логаритмична трансформация
df = df.withColumn("purchase_log", log1p("purchase_amount"))
df.show()

+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+------------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|day_of_week|  month|      purchase_log|
+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+------------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|     Sunday|January|  4.61512051684126|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January| 5.303304908059076|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January| 5.303304908059076|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|    Tuesday|January| 5.017279836814924|
|          4|    Anna Koleva|  -5|             50|     Electronics|   2025-01-08|  Wednesday|January|3.9318256327243257|
|          5|           NULL|  2

In [53]:
# Изчисляване на Q1 и Q3 (приблизително)
Q1_age, Q3_age = df.approxQuantile("age", [0.25, 0.75], 0.0)
Q1_purchase, Q3_purchase = df.approxQuantile("purchase_amount", [0.25, 0.75], 0.0)
IQR_age = Q3_age - Q1_age
IQR_purchase = Q3_purchase - Q1_purchase

# Маркиране на outliers
df = df.withColumn("age_outlier", when((col("age") < Q1_age - 1.5*IQR_age) | (col("age") > Q3_age + 1.5*IQR_age), 1).otherwise(0))
df = df.withColumn("purchase_outlier", when((col("purchase_amount") < Q1_purchase - 1.5*IQR_purchase) | 
                                           (col("purchase_amount") > Q3_purchase + 1.5*IQR_purchase), 1).otherwise(0))

df.show()

+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+------------------+-----------+----------------+
|customer_id|           name| age|purchase_amount|product_category|purchase_date|day_of_week|  month|      purchase_log|age_outlier|purchase_outlier|
+-----------+---------------+----+---------------+----------------+-------------+-----------+-------+------------------+-----------+----------------+
|          1|    Ivan Petrov|  25|            100|     Electronics|   2025-01-05|     Sunday|January|  4.61512051684126|          0|               0|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January| 5.303304908059076|          0|               0|
|          2|  Maria Ivanova|  30|            200|        Clothing|   2025-01-06|     Monday|January| 5.303304908059076|          0|               0|
|          3|Georgi Georgiev|  40|            150|           Books|   2025-01-07|    Tuesday|January